# Öğrenci 1 — Veri Hazırlama ve Zaman Serisi Yapısı (Colab)

Low Carbon London akıllı sayaç verisi → canonical Parquet → 100K/1M/5M kümeleri → zaman serisi analizi.

**Çalışma zamanı:** Colab'da `Runtime > Change runtime type` ile *High-RAM* önerilir (zorunlu değil).

## 0. Repo ve kurulum
Repo GitHub'daysa ilk satırı kendi adresinizle değiştirin. Değilse zip'i Colab'a yükleyip açın.

In [ ]:
import os
# Zip'i Colab'a yüklediyseniz (soldaki Dosyalar > yükle simgesi). Klasör zaten varsa yeniden açmaz.
if not os.path.exists('/content/elektrik-buyuk-veri'):
    !unzip -q elektrik-buyuk-veri.zip -d /content
# GitHub'dan: !git clone https://github.com/<kullanici>/elektrik-buyuk-veri.git /content/elektrik-buyuk-veri
%cd /content/elektrik-buyuk-veri
!pip install -q -r requirements.txt
!java -version

## 1. Veriyi Drive'dan al
London Datastore otomatik (script) indirmeye izin vermiyor. Bu yüzden:

1. Tarayıcıda https://data.london.gov.uk/dataset/smartmeter-energy-use-data-in-london-households sayfasını açın.
2. **low-carbon-london-data-168-files** (759 MB) dosyasını indirin.
3. İnen `Partitioned LCL Data.zip` dosyasını Google Drive'da **Drive'ım** ana klasörüne yükleyin.

Aşağıdaki hücre zip'i Drive'dan alır ve ilk 8 parçayı (≈ 8M satır) açar. Tam veri için `--max-files` kısmını silin.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SRC = '/content/drive/MyDrive/Partitioned LCL Data.zip'
assert os.path.exists(SRC), f'Bulunamadı: {SRC} — dosya adını/konumunu kontrol edin'
!mkdir -p data/raw && cp "{SRC}" data/raw/lcl_partitioned.zip
!python -m src.ogrenci1_veri_hazirlama.s01_download --variant partitioned --max-files 8
!ls -lh data/raw/lcl_csv | head
!head -3 data/raw/lcl_csv/*.csv | head -8

## 2. Ham CSV → canonical Parquet (PySpark)

In [ ]:
import os
os.environ['EBV_DRIVER_MEMORY'] = '8g'
!python -m src.ogrenci1_veri_hazirlama.s02_prepare
!cat metrics/s02_prepare.json

## 3. 100K / 1M / 5M alt kümeleri (CSV + Parquet)

In [ ]:
!python -m src.ogrenci1_veri_hazirlama.s03_make_subsets
import json, pandas as pd
pd.DataFrame(json.load(open('metrics/s03_subsets.json'))['subsets']).T

## 4. Zaman serisi yapısı analizi

In [ ]:
!python -m src.ogrenci1_veri_hazirlama.s04_ts_structure

In [ ]:
from IPython.display import Markdown, Image, display
import glob
display(Markdown(open('docs/ogrenci1_bulgular.md', encoding='utf-8').read().split('![')[0]))
for f in sorted(glob.glob('docs/figures/ogr1_*.png')):
    display(Image(f))

## 5. Ekibe teslim: çıktıları Google Drive'a kopyala
Bu hücre `MyDrive/ebv_data` klasörünü oluşturur. Sonra Drive'da bu klasöre sağ tıklayıp **Paylaş** diyerek ekiple paylaşın.

In [ ]:
# Drive zaten 1. adımda bağlandı
!mkdir -p /content/drive/MyDrive/ebv_data
!cp -r data/processed data/subsets docs metrics /content/drive/MyDrive/ebv_data/
!du -sh /content/drive/MyDrive/ebv_data/*
print('Tamam: Drive > MyDrive > ebv_data klasörünü ekiple paylaşın.')

## 6. ACORN düzeltmesi (ham veriyi yeniden indirmeden)
Güncel LCL dosyalarında Acorn kolonları yok. Bu hücre Drive'daki hazır veriyi alır, Kaggle'daki hane tablosuyla birleştirir, alt kümeleri ve analizi yeniden üretir, sonucu Drive'a geri yazar.

Önce: https://www.kaggle.com/datasets/jeanmidev/smart-meters-in-london sayfasından yalnızca **informations_households.csv** dosyasını indirip Drive'ım ana klasörüne yükleyin.

Bu bölüm tek başına çalışır: yeni bir Colab oturumunda önce **0. Repo ve kurulum** hücresini, sonra bunu çalıştırın.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/elektrik-buyuk-veri
D = '/content/drive/MyDrive'
!mkdir -p data/raw
!cp -r {D}/ebv_data/processed data/ && cp -r {D}/ebv_data/metrics . && cp {D}/informations_households.csv data/raw/
!python -m src.ogrenci1_veri_hazirlama.s02b_enrich_households
!python -m src.ogrenci1_veri_hazirlama.s03_make_subsets
!python -m src.ogrenci1_veri_hazirlama.s04_ts_structure
!rm -rf {D}/ebv_data/processed {D}/ebv_data/subsets {D}/ebv_data/docs {D}/ebv_data/metrics
!cp -r data/processed data/subsets docs metrics {D}/ebv_data/
!du -sh {D}/ebv_data/*